In [1]:
import requests
import pandas as pd
import io

def descargar_zona_cuadricula(fecha_inicio, fecha_fin, mag_min, min_lat, max_lat, min_lon, max_lon):
    url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
    params = {
        "format": "csv",
        "starttime": fecha_inicio,
        "endtime": fecha_fin,
        "minmagnitude": mag_min,
        # Filtros geográficos:
        "minlatitude": min_lat,
        "maxlatitude": max_lat,
        "minlongitude": min_lon,
        "maxlongitude": max_lon
    }
    
    response = requests.get(url, params=params)
    if response.status_code == 200:
        return pd.read_csv(io.StringIO(response.text))
    else:
        print(f"Error: {response.status_code}")
        return None

In [2]:
import time
# Definimos las coordenadas de la cuadrícula de California
min_lat_california = 32.0
max_lat_california = 42.5
min_lon_california = -125.0
max_lon_california = -114.0

# Usamos la magia de Pandas para crear una lista de fechas mes a mes
# MS significa "Month Start" (Inicio de mes)
fechas = pd.date_range(start="1990-01-01", end="2000-01-01", freq="MS")

lista_df = []

print("Iniciando extracción mes a mes...")

# Bucle sobre los meses
for i in range(len(fechas) - 1):
    inicio = fechas[i].strftime("%Y-%m-%d")
    fin = fechas[i+1].strftime("%Y-%m-%d")
    
    print(f"Descargando {inicio} a {fin}...", end=" ")
    
    df_mes = descargar_zona_cuadricula(
        inicio, fin, 2.5, 
        min_lat_california, max_lat_california, 
        min_lon_california, max_lon_california
    )
    
    if df_mes is not None:
        print(f"{len(df_mes)} sismos.")
        lista_df.append(df_mes)
        
    # Pausa de 0.5 segundos para respetar los servidores del USGS
    time.sleep(0.5)

# Unir todo el catálogo si se han descargado datos
if lista_df:
    df_california_total = pd.concat(lista_df, ignore_index=True)
    
    # Por seguridad, eliminamos duplicados (si un sismo cayó exactamente a las 00:00:00 del día de corte)
    df_california_total = df_california_total.drop_duplicates(subset='id')
    
    # Ordenamos de más antiguo a más reciente
    df_california_total = df_california_total.sort_values(by='time').reset_index(drop=True)
    
    print(f"\n¡Éxito Total! Se han descargado y unificado {len(df_california_total)} sismos de California.")
    
    # Guardar en CSV para tu TFM
    df_california_total.to_csv("catalogo_california_1990_2000.csv", index=False)

Iniciando extracción mes a mes...
Descargando 1990-01-01 a 1990-02-01... 147 sismos.
Descargando 1990-02-01 a 1990-03-01... 125 sismos.
Descargando 1990-03-01 a 1990-04-01... 179 sismos.
Descargando 1990-04-01 a 1990-05-01... 207 sismos.
Descargando 1990-05-01 a 1990-06-01... 132 sismos.
Descargando 1990-06-01 a 1990-07-01... 107 sismos.
Descargando 1990-07-01 a 1990-08-01... 80 sismos.
Descargando 1990-08-01 a 1990-09-01... 98 sismos.
Descargando 1990-09-01 a 1990-10-01... 125 sismos.
Descargando 1990-10-01 a 1990-11-01... 98 sismos.
Descargando 1990-11-01 a 1990-12-01... 107 sismos.
Descargando 1990-12-01 a 1991-01-01... 103 sismos.
Descargando 1991-01-01 a 1991-02-01... 80 sismos.
Descargando 1991-02-01 a 1991-03-01... 107 sismos.
Descargando 1991-03-01 a 1991-04-01... 173 sismos.
Descargando 1991-04-01 a 1991-05-01... 89 sismos.
Descargando 1991-05-01 a 1991-06-01... 76 sismos.
Descargando 1991-06-01 a 1991-07-01... 105 sismos.
Descargando 1991-07-01 a 1991-08-01... 94 sismos.
Desc

In [3]:
df_look_back = pd.read_csv("catalogo_california_1990_2000.csv")
df_look_back_eq = df_look_back[df_look_back["type"] == "earthquake"]
df_look_back_eq['lat_bin'] = df_look_back_eq['latitude'].round(0)
df_look_back_eq['lon_bin'] = df_look_back_eq['longitude'].round(0)
df_look_back_eq['cell_id'] = df_look_back_eq['lat_bin'].astype(str) + "_" + df_look_back_eq['lon_bin'].astype(str)

C:\Users\longj\AppData\Local\Temp\ipykernel_24004\1505586980.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_look_back_eq['lat_bin'] = df_look_back_eq['latitude'].round(0)
C:\Users\longj\AppData\Local\Temp\ipykernel_24004\1505586980.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_look_back_eq['lon_bin'] = df_look_back_eq['longitude'].round(0)
C:\Users\longj\AppData\Local\Temp\ipykernel_24004\1505586980.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataF

In [11]:
df_look_back_eq['time'] = pd.to_datetime(df_look_back_eq['time'])
df_look_back_eq['date'] = df_look_back_eq['time'].dt.floor('D')
daily_look_back = df_look_back_eq.groupby(['cell_id', 'date']).agg(
    eq_count=('mag', 'count')
).reset_index()

C:\Users\longj\AppData\Local\Temp\ipykernel_24004\2554251714.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_look_back_eq['time'] = pd.to_datetime(df_look_back_eq['time'])
C:\Users\longj\AppData\Local\Temp\ipykernel_24004\2554251714.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_look_back_eq['date'] = df_look_back_eq['time'].dt.floor('D')


In [13]:
daily_look_back["eq_count"].describe()

count    11051.000000
mean         2.129400
std          7.398465
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max        262.000000
Name: eq_count, dtype: float64

In [19]:
daily_look_back['date'] = pd.to_datetime(daily_look_back['date']).dt.date
all_dates_look_back = pd.date_range(start=daily_look_back['date'].min(), end=daily_look_back['date'].max(), freq='D')
all_dates_look_back = pd.to_datetime(all_dates_look_back).date
all_cells_look_back = daily_look_back['cell_id'].unique()
full_index = pd.MultiIndex.from_product([all_cells_look_back, all_dates_look_back], names=['cell_id', 'date'])

# Reindexamos y rellenamos los nulos (días sin sismos) con ceros
panel_look_back = daily_look_back.set_index(['cell_id', 'date']).reindex(full_index).reset_index()
panel_look_back['eq_count'] = panel_look_back['eq_count'].fillna(0)

In [20]:
panel_look_back["eq_count"].describe()

count    335984.000000
mean          0.070039
std           1.394441
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         262.000000
Name: eq_count, dtype: float64

In [21]:
panel_look_back = panel_look_back.sort_values(['cell_id', 'date'])
panel_look_back.to_csv('panel_look_back.csv', index=False)

In [22]:
panel_look_back["eq_count"].describe()

count    335984.000000
mean          0.070039
std           1.394441
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max         262.000000
Name: eq_count, dtype: float64